Crossmatching the 90% localisation volume of the simuated events with the mock galaxy catalogue.

The code was written by Mária Pálfi (marika97@student.elte.hu).

In [ ]:
import pandas as pd
import numpy as np
import h5py 

from astropy.coordinates import SkyCoord, ICRS, SphericalRepresentation
from astropy import units as u
from astropy.cosmology import FlatLambdaCDM

from ligo.skymap.io import read_sky_map
from ligo.skymap.postprocess import crossmatch

### Loading the galaxy catalog

In [ ]:
filename = "mock_for_sm_abs.hdf5"

data = pd.DataFrame( data = [] )
with h5py.File(filename, "r") as f:
    for a in f.keys():
        print(a)
        # Get the data
        if a == 'radec_lim':
            print('skip')
        else:
            data[str(a)] = list(f[a])

In [ ]:
# we will need to have luminosity distance
cosmo = FlatLambdaCDM(H0=73, Om0=0.25, Ob0=0.045)
dL = cosmo.luminosity_distance(data.z)
data['dL'] = np.array(dL)
data.head()

In [ ]:
len(data)

### Saving the coordinates as SkyCoord objects

In [ ]:
coordinates = SkyCoord( ra = data['ra']*180/np.pi*u.degree,
                        dec = data['dec']*180/np.pi*u.degree,
                        distance=data.dL*u.Mpc,
                        unit = [ 'deg', 'deg', 'Mpc'],
                        frame=ICRS,
                        representation_type = SphericalRepresentation )

### Searching for the possible host galaxies

In [ ]:
def find_host( eventskymap, outname ):
    skymap = read_sky_map( eventskymap, moc=True)
    result = crossmatch(skymap, coordinates)
    searched_prob_vol = result.searched_prob_vol[result.searched_prob_vol < 0.9 ] 
    hosts = pd.DataFrame( data[result.searched_prob_vol < 0.9] )
    hosts['searched_prob_vol'] = searched_prob_vol
    print( len(hosts) )
    hosts.to_csv( outname )

In [ ]:
for num in range(195):
    print( 'Event', num )
    find_host( './outdir_folder/bbh_injection_'+str(num)+'/result/bbh_injection_data0_0_analysis_H1L1V1_skymap.fits',
          'host_files_995/host'+str(num)+'.txt' )